# CF2 — Contact to Metriplectic Evolution (Code Recreation)

- Canon (anchor-only; do not duplicate): [CF2 — Contact to Metriplectic Evolution](../../Complete-Formalisms/CF2_Contact_to_Metriplectic_Evolution.md)
- Scope: exhaustive, testable code realization of CF2 with meters/gates. Notebook compressed into ≤4 top-level sections, implementing all worked subsections as numbered sub-items.

Canon anchors (no duplication):
- GENERIC evolution and degeneracies: [VDM-E-140](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-140), [VDM-E-142](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-142), [VDM-E-143](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-143)
- Algorithmic composition (Strang/JMJ): [VDM-A-036](../../../z.CANONICAL_Algorithms/00_ALGORITHMS.md#vdm-a-036)
- Validation gates: [Validation Metrics](../../../z.CANONICAL_Validation_Metrics/00_VALIDATION_METRICS.md)


## I. Primitives → Brackets (CF2 §1–2)

- 1.1 Contact manifold definition and 1-form α
- 1.2 Reeb vector field R checks
- 1.3 Legendre submanifolds (equilibria) — numerical check α|_{Legendre}=0
- 2.1 Contact Hamiltonian K and X_K
- 2.2 Contact bracket {·,·}_c
- 2.3 Evolution equations for a test K (harmonic form)


In [ ]:
import numpy as np
np.set_printoptions(precision=8, suppress=True)

# (1.1) Contact 1-form α = ds - p dq  on coordinates (q,p,s)
def alpha_components(q: float, p: float, s: float):
    # components in basis [dq, dp, ds]
    return np.array([-p, 0.0, 1.0], dtype=np.float64)

# dα = dq ∧ dp (antisymmetric)
def d_alpha_matrix():
    Omega = np.zeros((3,3), dtype=np.float64)
    Omega[0,1] = 1.0
    Omega[1,0] = -1.0
    return Omega

# (1.2) Reeb field R satisfies α(R)=1 and ι_R dα = 0
def reeb_vector():
    return np.array([0.0, 0.0, 1.0], dtype=np.float64)

# Quick wedge coefficient proxy for α ∧ dα volume (nonzero)
def wedge_alpha_dalpha_volume_coeff(alpha, Omega):
    return float(alpha[2]*Omega[0,1] + alpha[0]*Omega[1,2] + alpha[1]*Omega[2,0])

q,p,s = 0.3, -0.7, 0.0
alpha = alpha_components(q,p,s)
Omega = d_alpha_matrix()
R = reeb_vector()
checks = {
    'alpha_R': float(alpha @ R),                  # expect 1
    'iR_dalpha_norm': float(np.linalg.norm(Omega @ R)),  # expect 0
    'alpha_wedge_dalpha_coeff': wedge_alpha_dalpha_volume_coeff(alpha, Omega)  # expect ≠ 0
}
checks


### 1.3 Legendre submanifolds (equilibria): α|_{Legendre} = 0 (numeric)

We construct a thermodynamic potential $U(S,V)$ and verify that on the Legendre submanifold defined by $T=\partial U/\partial S$ and $p=-\partial U/\partial V$, the pullback of the contact form $\alpha=dU - T\,dS - p\,dV$ vanishes along arbitrary tangents $(dS, dV)$.

In [ ]:
# Smooth example: U(S,V) = A e^{S/C} + B V^{-γ}
A, C, B, gamma = 1.0, 1.0, 0.7, 1.0
def U_SV(S, V):
    return A*np.exp(S/C) + B * V**(-gamma)
def T_of(S, V):
    return (A/C)*np.exp(S/C)
def p_of(S, V):
    return gamma*B * V**(-(gamma+1))

def pullback_alpha_residual(S, V, dS, dV, h=1e-8):
    # Linearized dU via partials
    dU = ( (U_SV(S+h,V) - U_SV(S-h,V))/(2*h) )*dS + ( (U_SV(S,V+h) - U_SV(S,V-h))/(2*h) )*dV
    T = T_of(S,V); p = p_of(S,V)
    return float(dU - T*dS - p*dV)

rng = np.random.default_rng(42)
res = []
for _ in range(200):
    S0 = rng.uniform(-1.0, 2.0)
    V0 = rng.uniform(0.2, 3.0)
    dS = rng.normal(scale=0.1)
    dV = rng.normal(scale=0.1)
    res.append(abs(pullback_alpha_residual(S0,V0,dS,dV)))

{'Legendre_pullback_max|α|': float(np.max(res)), 'Legendre_pullback_med|α|': float(np.median(res)), 'pass_tol_1e-10': (np.max(res) < 1e-10)}


In [ ]:
# (2.1) Contact Hamiltonian vector field X_K in (q,p,s)
def dK_dq(K, q, p, s, h: float = 1e-6):
    return (K(q+h,p,s) - K(q-h,p,s)) / (2*h)
def dK_dp(K, q, p, s, h: float = 1e-6):
    return (K(q,p+h,s) - K(q,p-h,s)) / (2*h)

def X_contact(K, q, p, s):
    dq = dK_dp(K, q, p, s)
    dp = -dK_dq(K, q, p, s)
    ds = p*dK_dp(K, q, p, s) - K(q,p,s)
    return np.array([dq, dp, ds], dtype=np.float64)

# (2.2) Contact bracket {f,g}_c = ω(X_f,X_g) + f α(X_g) - g α(X_f)
def omega_bilinear(X, Y):
    return float((X.reshape(1,3) @ d_alpha_matrix() @ Y.reshape(3,1))[0,0])
def alpha_dot_of(X, q, p, s):
    return float(alpha_components(q,p,s) @ X)
def contact_bracket(f, g, q, p, s):
    Xf = X_contact(f, q, p, s)
    Xg = X_contact(g, q, p, s)
    return omega_bilinear(Xf, Xg) + f(q,p,s)*alpha_dot_of(Xg,q,p,s) - g(q,p,s)*alpha_dot_of(Xf,q,p,s)

# (2.3) Test antisymmetry on sample functions
omega0 = 1.0
K_H = lambda q,p,s,omega=omega0: 0.5*(p*p + (omega**2)*(q*q))
f = lambda q,p,s: q*p + 0.2*s + 0.1*q*q
g = lambda q,p,s: 0.5*q*q - 0.3*p*p + 0.1*s
q0,p0,s0 = 0.25, -0.4, 0.0
val_fg = contact_bracket(f,g,q0,p0,s0)
val_gf = contact_bracket(g,f,q0,p0,s0)
{'{f,g}_c': val_fg, '{g,f}_c': val_gf, 'antisym_ok': np.allclose(val_fg, -val_gf, atol=1e-10)}


## II. Thermodynamics Contact Structure (CF2 §3)

- 3.1 First law: α = dU − T dS − p dV on (U,T,S,p,V) — already verified via Legendre pullback
- 3.2 Free energy as contact Hamiltonian: compute $F(T,V)=\min_S [U(S,V)-TS]$ and verify $\partial F/\partial T=-S$, $\partial F/\partial V=-p$ (numeric)
- 3.3 Gibbs/Maxwell relation: from $F(T,V)$, verify $\partial S/\partial V|_T = \partial p/\partial T|_V$ (numeric)


In [ ]:
# 3.2/3.3: Construct F(T,V) by Legendre transform and verify identities
A, C, B, gamma = 1.0, 1.0, 0.7, 1.0
def U_SV(S, V):
    return A*np.exp(S/C) + B * V**(-gamma)
def T_of(S, V):
    return (A/C)*np.exp(S/C)
def p_of_SV(S, V):
    return gamma*B * V**(-(gamma+1))

def S_of_TV(T, V):
    # Inverse of T = (A/C) e^{S/C}
    return C*np.log(max(T,1e-300)*C/A)

def F_of_TV(T, V):
    S_star = S_of_TV(T, V)
    return U_SV(S_star, V) - T*S_star

def fd1(f, x, axis=0, h=1e-6):
    # Central diff for 2-var function; axis=0 for first arg, axis=1 for second
    if axis == 0:
        return (f(x[0]+h, x[1]) - f(x[0]-h, x[1]))/(2*h)
    else:
        return (f(x[0], x[1]+h) - f(x[0], x[1]-h))/(2*h)

def verify_free_energy_identities(T=2.0, V=1.7):
    S_star = S_of_TV(T,V)
    p_val  = p_of_SV(S_star, V)  # independent of S in this model
    dF_dT  = fd1(lambda t,v: F_of_TV(t,v), (T,V), axis=0)
    dF_dV  = fd1(lambda t,v: F_of_TV(t,v), (T,V), axis=1)
    return {
        'S*': float(S_star), 'p': float(p_val),
        'res_T': float(dF_dT + S_star),         # should be ~0
        'res_V': float(dF_dV + p_val),          # should be ~0
        'pass_T': abs(dF_dT + S_star) < 1e-10,
        'pass_V': abs(dF_dV + p_val) < 1e-10
    }

verify_free_energy_identities()


In [ ]:
# 3.3 Maxwell relation from F(T,V): ∂S/∂V|_T = ∂p/∂T|_V
def S_of_TV_viaF(T,V):
    return -fd1(lambda t,v: F_of_TV(t,v), (T,V), axis=0)
def p_of_TV_viaF(T,V):
    return -fd1(lambda t,v: F_of_TV(t,v), (T,V), axis=1)

def verify_maxwell(T=2.0, V=1.7):
    dS_dV_T = (S_of_TV_viaF(T, V+1e-5) - S_of_TV_viaF(T, V-1e-5))/(2e-5)
    dp_dT_V = (p_of_TV_viaF(T+1e-5, V) - p_of_TV_viaF(T-1e-5, V))/(2e-5)
    return {
        'residual_∂S/∂V|T_minus_∂p/∂T|V': float(dS_dV_T - dp_dT_V),
        'pass_tol_1e-10': abs(dS_dV_T - dp_dT_V) < 1e-10
    }

verify_maxwell()


## III. GENERIC mapping & decomposition (CF2 §4–5)

- 4.1 GENERIC framework and structure checks ([VDM-E-140](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-140))
- 4.2 Metriplectic degeneracies and projectors ([VDM-E-142](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-142))
- 4.3 Contact → GENERIC mapping residual (pointwise vector-field comparison)
- 5.1 Constructive decomposition E = K|_{s=s0}, S = p·∂K/∂p − K, canonical L, scalar M ≥ 0
- 5.2 Validation gates bundle (report-only here; thresholds live in canon)


In [ ]:
# Utilities for gradients/Hessians and structure checks
def grad2(f, q, p, h: float = 1e-6):
    dq = (f(q+h,p) - f(q-h,p)) / (2*h)
    dp = (f(q,p+h) - f(q,p-h)) / (2*h)
    return np.array([dq, dp], dtype=np.float64)

def hess2(f, q, p, h: float = 1e-4):
    f00 = f(q, p)
    fpp = (f(q, p+h) - 2*f00 + f(q, p-h)) / (h*h)
    fqq = (f(q+h, p) - 2*f00 + f(q-h, p)) / (h*h)
    fpq = (f(q+h, p+h) - f(q+h, p-h) - f(q-h, p+h) + f(q-h, p-h)) / (4*h*h)
    return np.array([[fqq, fpq],[fpq, fpp]], dtype=np.float64)

def projector_perp(v: np.ndarray) -> np.ndarray:
    n2 = float(v @ v)
    if n2 <= 1e-16:
        return np.eye(v.size, dtype=np.float64)
    return np.eye(v.size, dtype=np.float64) - np.outer(v, v)/n2

def check_structure(L: np.ndarray, M: np.ndarray) -> dict:
    antisym_ok = np.allclose(L.T, -L, atol=1e-12)
    sym_ok    = np.allclose(M.T,  M, atol=1e-12)
    eigM      = np.linalg.eigvalsh(M)
    psd_ok    = (np.min(eigM) >= -1e-12)
    return {'L_antisym': antisym_ok, 'M_sym': sym_ok, 'M_min_eig': float(np.min(eigM)), 'M_psd': psd_ok}

# Constructive decomposition from K (5.1)
omega = 1.0
K = lambda q,p,s: 0.5*(p*p + (omega**2)*(q*q))
s0 = 0.0
E = lambda q,p: K(q,p,s0)
S = lambda q,p: p*dK_dp(K,q,p,s0) - K(q,p,s0)

def L_canonical():
    return np.array([[0.0, 1.0],[-1.0, 0.0]], dtype=np.float64)
def M_scalar(g: float):
    return g * np.eye(2, dtype=np.float64)

# Positive scalar from (minus) Hessian clamps of S (Ruppeiner-inspired local scale)
def M_from_S_local(q, p):
    Hs = hess2(S, q, p)
    g_scalar = float(max(1e-12, 0.5*max(0.0, -Hs[0,0]) + 0.5*max(0.0, -Hs[1,1])))
    return M_scalar(g_scalar)

def pb(L: np.ndarray, gradA: np.ndarray, gradB: np.ndarray) -> float:
    return float(gradA @ (L @ gradB))
def mb(M: np.ndarray, gradA: np.ndarray, gradB: np.ndarray) -> float:
    return float(gradA @ (M @ gradB))

# Pointwise mapping residual between contact X_K and metriplectic v = L∇E + M∇S (qp components)
def mapping_vectors(q, p):
    egrad = grad2(E, q, p)
    sgrad = grad2(S, q, p)
    L = L_canonical()
    M = M_from_S_local(q,p)
    PS, PE = projector_perp(sgrad), projector_perp(egrad)
    Lc, Mc = PS @ L @ PS, PE @ M @ PE
    XK = X_contact(K, q, p, s0)
    v_con = np.array([XK[0], XK[1]], dtype=np.float64)
    v_gen = (Lc @ egrad) + (Mc @ sgrad)
    return v_con, v_gen, Lc, Mc, egrad, sgrad

def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    yt_mean = np.mean(y_true)
    ss_res = float(np.sum((y_true - y_pred)**2))
    ss_tot = float(np.sum((y_true - yt_mean)**2) + 1e-30)
    return 1.0 - ss_res/ss_tot

def sample_grid(n: int = 21, lim: float = 1.0):
    xs = np.linspace(-lim, lim, n)
    pts = [(q,p) for q in xs for p in xs]
    return pts

def mapping_metrics(n: int = 21, lim: float = 1.0):
    Vc = []  # contact vectors
    Vg = []  # generic vectors
    for (q,p) in sample_grid(n, lim):
        v_con, v_gen, *_ = mapping_vectors(q,p)
        Vc.append(v_con)
        Vg.append(v_gen)
    Vc = np.asarray(Vc)
    Vg = np.asarray(Vg)
    residuals = np.linalg.norm(Vg - Vc, axis=1)
    r2_x = r2_score(Vc[:,0], Vg[:,0])
    r2_y = r2_score(Vc[:,1], Vg[:,1])
    r2_all = r2_score(Vc.reshape(-1), Vg.reshape(-1))
    return {
        'grid_size': int(n*n),
        'residual_norm_min': float(np.min(residuals)),
        'residual_norm_med': float(np.median(residuals)),
        'residual_norm_max': float(np.max(residuals)),
        'R2_x': float(r2_x), 'R2_y': float(r2_y), 'R2_all': float(r2_all)
    }

def degeneracy_metrics(n: int = 21, lim: float = 1.0):
    L = L_canonical()
    pb_SE_raw = []
    mb_ES_raw = []
    pb_SE_cond = []
    mb_ES_cond = []
    for (q,p) in sample_grid(n, lim):
        e = grad2(E,q,p)
        s = grad2(S,q,p)
        M = M_from_S_local(q,p)
        pb_SE_raw.append(abs(pb(L,s,e)))
        mb_ES_raw.append(abs(mb(M,e,s)))
        PS, PE = projector_perp(s), projector_perp(e)
        Lc, Mc = PS @ L @ PS, PE @ M @ PE
        pb_SE_cond.append(abs(pb(Lc,s,e)))
        mb_ES_cond.append(abs(mb(Mc,e,s)))
    return {
        'max{|{S,E}_J|}_raw': float(np.max(pb_SE_raw)),
        'max{|(E,S)_M|}_raw': float(np.max(mb_ES_raw)),
        'max{|{S,E}_J|}_cond': float(np.max(pb_SE_cond)),
        'max{|(E,S)_M|}_cond': float(np.max(mb_ES_cond))
    }

def entropy_monotonicity_metrics(n: int = 21, lim: float = 1.0, dts=(1e-3,5e-3,1e-2)):
    # M-only step: x -> x + dt * (M @ ∇S), projected to enforce (E,·)_M=0
    results = {}
    pts = sample_grid(n, lim)
    for dt in dts:
        ok = 0
        for (q,p) in pts:
            e = grad2(E,q,p)
            s = grad2(S,q,p)
            M = M_from_S_local(q,p)
            PE = projector_perp(e)
            Mc = PE @ M @ PE
            x = np.array([q,p], dtype=np.float64)
            x1 = x + dt*(Mc @ s)
            dS = float(S(x1[0], x1[1]) - S(x[0], x[1]))
            ok += (dS >= -1e-14)
        results[f'dt={dt}'] = ok/len(pts)
    return results

# Structure sanity
check_structure(L_canonical(), M_scalar(0.1))


In [ ]:
# Grid-level metrics summary (report-only; thresholds are canon-owned)
mm = mapping_metrics(n=25, lim=1.0)
dm = degeneracy_metrics(n=25, lim=1.0)
em = entropy_monotonicity_metrics(n=25, lim=1.0)
report = {'mapping_metrics': mm, 'degeneracy_metrics': dm, 'entropy_monotonicity': em}
report


## IV. Worked Example + Validation (CF2 §6, §9)

- 6.1–6.5 Ideal gas minimal example (numeric toy)
- 9.1 Mathematical consistency checks (structure/degeneracy)
- 9.2 Physical consistency checks (entropy trend)
- 9.3 Numerical validation (grid summaries)


In [ ]:
N = 1.0e23
kB = 1.380649e-23
T0 = 300.0

# Coordinates (V,P) with canonical Poisson {f,g} = f_V g_P - f_P g_V
def grad_VP(f, V, P, h=1e-6):
    dV = (f(V+h,P) - f(V-h,P))/(2*h)
    dP = (f(V,P+h) - f(V,P-h))/(2*h)
    return np.array([dV, dP], dtype=np.float64)
def pb_VP(gradA, gradB):
    return float(gradA[0]*gradB[1] - gradA[1]*gradB[0])

# Ideal gas toy (illustrative only)
E_id = lambda V,P: 1.5 * N * kB * T0                  # independent of (V,P)
S_id = lambda V,P: N*kB*(np.log(max(V,1e-12)) + 2.5)  # monotone in V
L_vp = np.array([[0.0, 1.0],[-1.0, 0.0]])
M_vp = 0.1*np.eye(2)

V0,P0 = 1.0, 1.0e5
gV = grad_VP(S_id, V0, P0)
gE = grad_VP(E_id, V0, P0)
deg = {
    '{S,E}_J': pb_VP(gV, gE),
    '(E,S)_M': float(gE @ (M_vp @ gV))
}
# Entropy monotonicity under M-only
x = np.array([V0,P0])
x1 = x + 0.02*(M_vp @ gV)
dS = float(S_id(x1[0], x1[1]) - S_id(x[0], x[1]))
{'degeneracies': deg, 'ΔS_Monly_ideal': dS}


### Advanced topics & connections (links only; no duplication)
- 7.1 Non-equilibrium contact thermodynamics — see CF2 §7.1
- 7.2 Finite-time thermodynamics — see CF2 §7.2
- 7.3 Information geometry link (CF1 → CF2) — see CF2 §7.3; also [VDM-A-038](../../../z.CANONICAL_Algorithms/00_ALGORITHMS.md#vdm-a-038)
- 8.1–8.3 Connections to VDM unification — see CF2 §8; equations registry and T0 spec links
- 10.1–10.2 Open questions & next steps — see CF2 §10


### Repro notes and I/O policy
- Determinism: pure NumPy; IEEE‑754 double precision assumed.
- No file outputs here; production artifacts must route via [io_paths.py](../../../code/common/io_paths.py).

In [ ]:
# io_paths bootstrap (optional, no file writes in this notebook)
from pathlib import Path
import sys
COMMON = Path.cwd().resolve() / 'Derivation' / 'code' / 'common'
if COMMON.exists() and str(COMMON) not in sys.path:
    sys.path.insert(0, str(COMMON))
try:
    from io_paths import figure_path, log_path  # noqa: F401
except Exception as e:
    print('[warn] io_paths not available:', e)
